# JSONBench walkthrough

Generate a small travel dataset in memory.

In [ ]:
from json_bench.data import generate_travel_dataset

dataset = generate_travel_dataset(
    scale=0.01,
    seed=23,
    reference_date="2025-11-01",
)

## Deploy the databases

Start Docker and run `uv sync --locked` first. BONDI waits for each database to be ready.
Couchbase uses Community Edition with a `travel` bucket. Change the names or ports if needed.

In [ ]:
from bondi.deployment import deploy_couchbase, deploy_mongodb

mongo_deployment = deploy_mongodb(name="jsonbench-travel", port=27127)
couchbase_deployment = deploy_couchbase(
    name="jsonbench-travel-cb",
    management_port=18291,
    query_port=18293,
    kv_port=21230,
    bucket="travel",
)
print(mongo_deployment.uri)
print(couchbase_deployment.console_url)

## Load the same dataset into both databases

Load each collection into MongoDB and Couchbase, preserving document IDs and date strings.
BONDI derives Couchbase keys from the IDs and creates no query indexes.
The default `create` mode requires empty collections. Run cleanup before repeating deployment and loading.

In [ ]:
from bondi.couchbase import CouchbaseCollectionRef
from bondi.data import load_documents
from bondi.mongodb import MongoCollectionRef

for name, documents in dataset.items():
    mongo_result = load_documents(
        documents,
        target=mongo_deployment,
        collection=MongoCollectionRef("travel", name),
    )
    couchbase_result = load_documents(
        documents,
        target=couchbase_deployment,
        collection=CouchbaseCollectionRef("travel", "_default", name),
    )
    print(mongo_result.destinations, couchbase_result.destinations)

## Translate Q1

Group all bookings by tour ID and sum their stored total prices.
These functions return BONDI query objects without executing them.

In [ ]:
from pprint import pprint

from json_bench.queries import translate_couchbase, translate_mongodb

q1 = {
    "name": "Query 1",
    "description": "Total booking value per tour",
    "primary_collection": "bookings",
    "operations": {"group": {"by": "tour_id", "aggregates": {"total_revenue": "sum(total_price)"}}},
}

mongo_query = translate_mongodb(q1, database="travel")
couchbase_query = translate_couchbase(q1, bucket="travel", scope="_default")

pprint(mongo_query.pipeline)
print(couchbase_query.statement)
print(couchbase_query.query_context)

## Clean up when finished

Remove both deployments, including their data and saved configuration.

In [ ]:
mongo_deployment.remove()
couchbase_deployment.remove()